**Making HTTP request to the Amazon**

In [1]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US, en;q=0.5"
}

def get_reviews(url):
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print("Failed to fetch page")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    reviews = []

    review_blocks = soup.find_all("div", {"data-hook": "review"})

    for review in review_blocks:
        try:
            title = review.find("a", {"data-hook": "review-title"}).text.strip()
        except:
            title = None

        try:
            rating = review.find("i", {"data-hook": "review-star-rating"}).text.split()[0]
        except:
            rating = None

        try:
            body = review.find("span", {"data-hook": "review-body"}).text.strip()
        except:
            body = None

        reviews.append({
            "title": title,
            "rating": float(rating) if rating else None,
            "review_text": body
        })

    return reviews


if __name__ == "__main__":
    url = "https://www.amazon.com/product-reviews/PRODUCT_ID"

    all_reviews = []

    for page in range(1, 4):  # limited pages to avoid blocking
        paginated_url = url + f"?pageNumber={page}"
        print(f"Scraping page {page}...")

        data = get_reviews(paginated_url)
        all_reviews.extend(data)

        time.sleep(2)  # avoid rate limiting

    print(f"Total reviews scraped: {len(all_reviews)}")

Scraping page 1...
Failed to fetch page
Scraping page 2...
Failed to fetch page
Scraping page 3...
Failed to fetch page
Total reviews scraped: 0


**PreProcessing Pipeline**

In [2]:
import re
import pandas as pd

def clean_text(text):
    if not text:
        return ""

    # remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # lowercase
    text = text.lower()

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def remove_duplicates(df):
    return df.drop_duplicates(subset=["review_text"])


def handle_missing_values(df):
    df = df.dropna(subset=["review_text", "rating"])
    return df


def preprocess_data(reviews):
    df = pd.DataFrame(reviews)

    # handle missing values
    df = handle_missing_values(df)

    # clean text
    df["cleaned_review"] = df["review_text"].apply(clean_text)

    # remove duplicates
    df = remove_duplicates(df)

    return df


if __name__ == "__main__":
    # Example usage
    sample_reviews = [
        {"title": "Great!", "rating": 5, "review_text": "<b>Awesome product!!</b>"},
        {"title": "Bad", "rating": 1, "review_text": "Terrible experience..."},
        {"title": "Great!", "rating": 5, "review_text": "<b>Awesome product!!</b>"}
    ]

    df = preprocess_data(sample_reviews)
    print(df.head())

    title  rating               review_text       cleaned_review
0  Great!       5  <b>Awesome product!!</b>      awesome product
1     Bad       1    Terrible experience...  terrible experience
